# 크라베오 AI — q4_0 변환 (재학습 없음)
# Cravveo AI — q4_0 Conversion (No Retraining)

**목적 | Purpose:**
- q8_0 (3.4GB) → q4_0 (약 1.7GB)
- HuggingFace 기존 LoRA 가중치 재사용, 재학습 없음

**셀 순서 | Steps:**
1. 라이브러리 설치
2. CUDA 오류 방지 + HF에서 모델 직접 로드
3. 빠른 테스트
4. q4_0 변환 + 다운로드

In [ ]:
# 셀 1: 라이브러리 설치
# Cell 1: Install libraries
!pip install unsloth

In [ ]:
# 셀 2: CUDA 오류 방지 + 모델 로드
# Cell 2: CUDA fix + load model
#
# 핵심: HuggingFace에 올라간 LoRA 모델을 직접 주소로 불러옴
# FastLanguageModel이 베이스 모델 + LoRA 가중치를 자동으로 합쳐줌

import ctypes, glob

paths = (
    glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib/libnvJitLink*.so*") +
    glob.glob("/usr/lib/**/libnvJitLink*.so*", recursive=True) +
    glob.glob("/usr/local/cuda*/**/libnvJitLink*.so*", recursive=True)
)
if paths:
    ctypes.CDLL(paths[0], mode=ctypes.RTLD_GLOBAL)
    print(f"CUDA fix 적용 | CUDA fix applied: {paths[0]}")

from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="cravveo/cravveo-llama-lora",  # HF에 있는 LoRA 모델 주소
    max_seq_length=1024,
    load_in_4bit=True,
)
print("모델 로드 완료 | Model loaded")

In [ ]:
# 셀 3: 변환 전 테스트
# Cell 3: Test before conversion
FastLanguageModel.for_inference(model)

inputs = tokenizer(["질문: 크라베오 컴퍼니가 뭐야?\n답변: "], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=80, temperature=0.2)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# 셀 4: q4_0 변환 + 다운로드
# Cell 4: Convert to q4_0 + download
#
# q4_0 = 숫자 1개당 4비트로 압축 → 파일 크기 절반, CPU 부담 절반

model.save_pretrained_gguf(
    "cravveo-v3-gguf",
    tokenizer,
    quantization_method="q4_0"
)

from google.colab import files
import os

folder = "cravveo-v3-gguf_gguf"
gguf_file = [f for f in os.listdir(folder) if f.endswith(".gguf")][0]
size_gb = os.path.getsize(f"{folder}/{gguf_file}") / 1e9

print(f"파일명: {gguf_file}")
print(f"파일 크기: {size_gb:.2f} GB  (q8_0은 3.4GB였음)")

files.download(f"{folder}/{gguf_file}")
print("다운로드 시작 | Download started")